In [14]:
import torch
import os
import random
import torchvision.transforms as transforms
from PIL import Image
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
import torch.nn as nn

import numpy as np
import torch.optim as optim
import torch.nn.functional as F
from einops import rearrange
import time

In [15]:
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


Using device: cuda


In [16]:
img_size = (512, 512)

transform_img = transforms.Compose([
    transforms.Resize(img_size),
    transforms.ToTensor(),
])

transform_mask = transforms.Compose([
    transforms.Resize(img_size, interpolation=Image.NEAREST),
    transforms.ToTensor(),
])
class RetinalDataset(Dataset):
    def __init__(self, input_paths, target_paths, img_transform, mask_transform):
        self.input_paths = input_paths
        self.img_transform = img_transform
        self.target_paths = target_paths
        self.mask_transform = mask_transform

    def __len__(self):
        return len(self.input_paths)

    def __getitem__(self, idx):
        img = Image.open(self.input_paths[idx]).convert("RGB")
        img = self.img_transform(img)

        mask = Image.open(self.target_paths[idx]).convert("L")
        mask = self.mask_transform(mask)         # shape: [1, H, W], float
        mask = (mask > 0).long().squeeze(0)      # convert to [H, W] with int labels: 0 or 1

        return img, mask

    
    
def get_image_paths(img_dir, mask_dir, img_ext=".tif", mask_ext=".tif"):
    input_paths = sorted([
        os.path.join(img_dir, fname)
        for fname in os.listdir(img_dir)
        if fname.endswith(img_ext)
    ])
    target_paths = sorted([
        os.path.join(mask_dir, fname)
        for fname in os.listdir(mask_dir)
        # if fname.endswith(mask_ext)
        if fname.endswith(mask_ext)  # Assuming masks are in JPG format
    ])
    assert len(input_paths) == len(target_paths), "Mismatch between input and target image counts."
    return input_paths, target_paths





In [17]:


base_dir = "/home/cse/Documents/NP/nourin/GNN/DRIVE"
batch_size = 4


train_dir = os.path.join(base_dir, "training")
train_img_dir = os.path.join(train_dir, "aug_images")
train_mask_dir = os.path.join(train_dir, "aug_1st_manual")

test_dir = os.path.join(base_dir, "test")
test_img_dir = os.path.join(test_dir, "images")
test_mask_dir = os.path.join(test_dir, "1st_manual")

input_img_paths, target_img_paths = get_image_paths(img_dir = train_img_dir, mask_dir=train_mask_dir, img_ext="png", mask_ext="png")
test_img_paths, test_mask_paths = get_image_paths(img_dir = test_img_dir, mask_dir=test_mask_dir, img_ext="tif", mask_ext="tif")

# Shuffle and split
combined = list(zip(input_img_paths, target_img_paths))
random.seed(42)
random.shuffle(combined)
input_img_paths, target_img_paths = zip(*combined)
split_idx = int(0.9 * len(input_img_paths))

train_dataset = RetinalDataset(input_paths=input_img_paths[:split_idx], target_paths=target_img_paths[:split_idx], img_transform=transform_img, mask_transform= transform_mask)

val_dataset = RetinalDataset(input_paths=input_img_paths[split_idx:], target_paths=target_img_paths[split_idx:], img_transform=transform_img, mask_transform=transform_mask)

test_dataset = RetinalDataset(input_paths=test_img_paths, target_paths=test_mask_paths, img_transform=transform_img, mask_transform=transform_mask)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=0)

In [18]:


class conv_block(nn.Module):
    def __init__(self, in_c, out_c):
        super().__init__()

        self.conv1 = nn.Conv2d(in_c, out_c, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(out_c)

        self.conv2 = nn.Conv2d(out_c, out_c, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(out_c)

        self.relu = nn.ReLU()

    def forward(self, inputs):
        x = self.conv1(inputs)
        x = self.bn1(x)
        x = self.relu(x)

        x = self.conv2(x)
        x = self.bn2(x)
        x = self.relu(x)

        return x

class encoder_block(nn.Module):
    def __init__(self, in_c, out_c):
        super().__init__()

        self.conv = conv_block(in_c, out_c)
        self.pool = nn.MaxPool2d((2, 2))

    def forward(self, inputs):
        x = self.conv(inputs)
        p = self.pool(x)

        return x, p

class decoder_block(nn.Module):
    def __init__(self, in_c, out_c):
        super().__init__()

        self.up = nn.ConvTranspose2d(in_c, out_c, kernel_size=2, stride=2, padding=0)
        self.conv = conv_block(out_c+out_c, out_c)

    def forward(self, inputs, skip):
        x = self.up(inputs)
        x = torch.cat([x, skip], axis=1)
        x = self.conv(x)
        return x

class build_unet(nn.Module ):
    def __init__(self):
        super().__init__()
        self.name = "UnetModel"
        """ Encoder """
        self.e1 = encoder_block(3, 64)
        self.e2 = encoder_block(64, 128)
        self.e3 = encoder_block(128, 256)
        self.e4 = encoder_block(256, 512)

        """ Bottleneck """
        self.b = conv_block(512, 1024)

        """ Decoder """
        self.d1 = decoder_block(1024, 512)
        self.d2 = decoder_block(512, 256)
        self.d3 = decoder_block(256, 128)
        self.d4 = decoder_block(128, 64)

        """ Classifier """
        self.outputs = nn.Conv2d(64, 2, kernel_size=1, padding=0)

    def forward(self, inputs):
        """ Encoder """
        s1, p1 = self.e1(inputs)
        s2, p2 = self.e2(p1)
        s3, p3 = self.e3(p2)
        s4, p4 = self.e4(p3)

        """ Bottleneck """
        b = self.b(p4)

        """ Decoder """
        d1 = self.d1(b, s4)
        d2 = self.d2(d1, s3)
        d3 = self.d3(d2, s2)
        d4 = self.d4(d3, s1)

        outputs = self.outputs(d4)

        return outputs

In [19]:
class DiceLoss(nn.Module):
    def __init__(self, smooth=1):
        super(DiceLoss, self).__init__()
        self.smooth = smooth

    def forward(self, inputs, targets):

        inputs = torch.softmax(inputs, dim=1)  # [B, C, H, W]
        targets_one_hot = F.one_hot(targets, num_classes=inputs.shape[1]).permute(0, 3, 1, 2).float()

        intersection = (inputs * targets_one_hot).sum(dim=(2, 3))
        union = inputs.sum(dim=(2, 3)) + targets_one_hot.sum(dim=(2, 3))
        dice = (2 * intersection + self.smooth) / (union + self.smooth)
        return 1 - dice.mean()
    
class ComboLoss(nn.Module):
    def __init__(self, ce_weight=0.5):
        super(ComboLoss, self).__init__()
        self.ce = nn.CrossEntropyLoss(label_smoothing=0.1)
        self.ce = nn.CrossEntropyLoss()
        
        self.dice = DiceLoss()
        self.ce_weight = ce_weight

    def forward(self, inputs, targets):
        ce_loss = self.ce(inputs, targets)
        dice_loss = self.dice(inputs, targets)
        return self.ce_weight * ce_loss + (1 - self.ce_weight) * dice_loss

def compute_mean_iou(preds, labels, num_classes=2):
    preds = torch.argmax(preds, dim=1)  # [B, H, W]
    # print("Unique preds:", preds.unique())
    # print("Unique labels:", labels.unique())
    ious = []

    for cls in range(num_classes):
        pred_inds = (preds == cls)
        label_inds = (labels == cls)
        intersection = (pred_inds & label_inds).sum().float()
        union = (pred_inds | label_inds).sum().float()
        # print(f"[Class {cls}] Intersection: {intersection.item()}, Union: {union.item()}")
        if union == 0:
            continue  # skip this class
        ious.append(intersection / union)

    if len(ious) == 0:
        return torch.tensor(0.0, device=preds.device)

    mean_iou = torch.mean(torch.stack(ious))
    # print(f"Mean IoU: {mean_iou.item()}")
    return mean_iou


def compute_mean_iou_single(pred, target, num_classes=2):
    ious = []
    for cls in range(num_classes):
        pred_inds = (pred == cls)
        target_inds = (target == cls)
        intersection = (pred_inds & target_inds).sum().item()
        union = (pred_inds | target_inds).sum().item()
        if union == 0:
            continue
        ious.append(intersection / union)
    if len(ious) == 0:
        return 0.0
    return sum(ious) / len(ious)



In [20]:
def train(model, train_loader, val_loader, epochs, learning_rate, device, patience=500, version='XX', path='save_models', patching=False, patch_size=128):
    criterion = ComboLoss(ce_weight=0.4)
    optimizer = optim.Adam(model.parameters(), lr=learning_rate)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

    epochs_no_improve = 0
    val_losses, val_ious = [np.inf], [-np.inf]

    for epoch in range(epochs):
        model.train()
        train_loss, correct, total, iou_sum = 0, 0, 0, 0
        start_time = time.time()

        for images, masks in train_loader:
            images = images.to(device).float()
            masks = masks.to(device).squeeze(1).long()

            optimizer.zero_grad()
            outputs = model(images)
            assert masks.min() >= 0 and masks.max() < outputs.shape[1], \
                f"Invalid mask values: min={masks.min().item()}, max={masks.max().item()}, num_classes={outputs.shape[1]}"
            loss = criterion(outputs, masks)

            loss.backward()
            optimizer.step()
            train_loss += loss.item()

            preds = torch.argmax(outputs, dim=1)
            correct += (preds == masks).sum().item()
            total += masks.numel()
            iou_sum += compute_mean_iou(outputs, masks)

        train_acc = correct / total
        train_iou = iou_sum / len(train_loader)

        val_acc, val_iou, val_loss = validation(model, val_loader, device, show_metrics=False)
        print(f"Epoch {epoch+1}/{epochs} ({time.time() - start_time:.2f}s): acc: {train_acc:.4f}, loss: {train_loss / len(train_loader):.4f}, iou: {train_iou:.4f} | val_acc: {val_acc:.4f}, val_loss: {val_loss:.4f}, val_iou: {val_iou:.4f}")

        # Early stopping
        if val_iou > max(val_ious):
            epochs_no_improve = 0
            val_ious.append(val_iou)
            torch.save(model.state_dict(), f"{path}/{model.name}_{version}.pth")
        else:
            epochs_no_improve += 1
        if epochs_no_improve >= patience:
            print(f"Early stopping at epoch {epoch+1} with patience {patience}.")
            break

    model.load_state_dict(torch.load(f"{path}/{model.name}_{version}.pth"))


def validation(model, test_loader, device, patching=False, patch_size=128, show_metrics=True):
    model.eval()
    criterion = ComboLoss(ce_weight=0.4)
    val_loss, correct, total, iou_sum = 0.0, 0, 0, 0

    with torch.no_grad():
        for images, masks in test_loader:
            images = images.to(device).float()
            masks = masks.to(device).squeeze(1).long()

            outputs = model(images)
            assert masks.min() >= 0 and masks.max() < outputs.shape[1], \
                f"Invalid mask values: min={masks.min().item()}, max={masks.max().item()}, num_classes={outputs.shape[1]}"
            loss = criterion(outputs, masks)

            val_loss += loss.item()
            preds = torch.argmax(outputs, dim=1)
            correct += (preds == masks).sum().item()
            total += masks.numel()
            iou_sum += compute_mean_iou(outputs, masks)

    val_acc = correct / total
    avg_loss = val_loss / len(test_loader)
    val_iou = iou_sum / len(test_loader)

    if show_metrics:
        print(f"Accuracy: {val_acc:.4f}, Val_loss: {avg_loss:.4f}, Mean IoU: {val_iou:.4f}")

    return val_acc, val_iou, avg_loss


In [21]:

def tensor_to_pil(tensor):
    arr = tensor.detach().cpu().numpy()
    if arr.ndim == 3:
        arr = np.transpose(arr, (1, 2, 0)) 
        arr = (arr * 255).clip(0, 255).astype(np.uint8)
        return Image.fromarray(arr)
    elif arr.ndim == 2:
        arr = (arr * 255).clip(0, 255).astype(np.uint8)
        return Image.fromarray(arr)
    else:
        raise ValueError("Invalid tensor shape for conversion to image")

def prediction(model, test_loader, device, save_dir='images', img_name='predictions.png'):
    model.eval()
    triplet_rows = []
    count = 0
    cnt = 0
    
    for i, (image_tensor, mask_tensor) in enumerate(test_loader):
        if(cnt > 30):
            break
        cnt = cnt + 1
        image_tensor = image_tensor.to(device)  # [B, 3, H, W]
        mask_tensor = mask_tensor.to(device)    # [B, 1, H, W]

        B, C, H, W = image_tensor.shape

        with torch.no_grad():
            output = model(image_tensor) 
            preds = torch.argmax(output, dim=1).cpu() 

        for b in range(image_tensor.size(0)):
            input_img = tensor_to_pil(image_tensor[b])
            mask_img = tensor_to_pil(mask_tensor[b])
            pred_img = tensor_to_pil(preds[b])
            

            mask_img = mask_img.convert("RGB")
            pred_img = pred_img.convert("RGB")

            triplet = Image.new("RGB", (input_img.width * 3, input_img.height))
            triplet.paste(input_img, (0, 0))
            triplet.paste(mask_img, (input_img.width, 0))
            triplet.paste(pred_img, (input_img.width * 2, 0))

            triplet_rows.append(triplet)
            count += 1

    if triplet_rows:
        grid_width = triplet_rows[0].width
        grid_height = sum(img.height for img in triplet_rows)
        final_image = Image.new("RGB", (grid_width, grid_height))

        y_offset = 0
        for row in triplet_rows:
            final_image.paste(row, (0, y_offset))
            y_offset += row.height

        os.makedirs(save_dir, exist_ok=True)
        save_path = os.path.join(save_dir, f'{img_name}')
        final_image.save(save_path)
        print(f"Saved full grid image to {save_path}")
    else:
        print("No images to display.")



In [22]:
img_size = (512, 512) 
model = build_unet()
model = model.float().to(device) 
epochs = 50

train(model = model, train_loader=train_loader, val_loader=val_loader, epochs= epochs, learning_rate= 5e-3, device=device, patience=500, version=f'GNN_{epochs}', path='save_models', patching=False)

validation(model=model, test_loader=test_loader, device=device, patching=False, show_metrics=True)

Epoch 1/50 (16.82s): acc: 0.8550, loss: 0.4088, iou: 0.4819 | val_acc: 0.8563, val_loss: 1.4629, val_iou: 0.4355
Epoch 2/50 (16.79s): acc: 0.9078, loss: 0.2868, iou: 0.6353 | val_acc: 0.3471, val_loss: 5.0005, val_iou: 0.2081
Epoch 3/50 (16.85s): acc: 0.9214, loss: 0.2380, iou: 0.6856 | val_acc: 0.9000, val_loss: 0.3070, val_iou: 0.5880
Epoch 4/50 (16.88s): acc: 0.9341, loss: 0.2006, iou: 0.7286 | val_acc: 0.9155, val_loss: 0.2736, val_iou: 0.6991
Epoch 5/50 (16.94s): acc: 0.9357, loss: 0.1943, iou: 0.7366 | val_acc: 0.9262, val_loss: 0.2305, val_iou: 0.7019
Epoch 6/50 (16.92s): acc: 0.9380, loss: 0.1866, iou: 0.7467 | val_acc: 0.4754, val_loss: 1.3201, val_iou: 0.2985
Epoch 7/50 (16.92s): acc: 0.9419, loss: 0.1719, iou: 0.7633 | val_acc: 0.7868, val_loss: 0.4125, val_iou: 0.5558
Epoch 8/50 (16.95s): acc: 0.9417, loss: 0.1710, iou: 0.7633 | val_acc: 0.9347, val_loss: 0.1955, val_iou: 0.7398
Epoch 9/50 (16.95s): acc: 0.9401, loss: 0.1756, iou: 0.7582 | val_acc: 0.9239, val_loss: 0.2448,

/tmp/ipykernel_86732/1907151389.py:50: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(f"{path}/{model.name}_{version}.pth"))


Accuracy: 0.9497, Val_loss: 0.1541, Mean IoU: 0.7717


(0.9496877670288086, tensor(0.7717, device='cuda:0'), 0.1540513128042221)

In [23]:
prediction(model=model, test_loader=test_loader, device=device, img_name=f'GNN_{epochs}.png')

Saved full grid image to images/GNN_50.png
